In [ ]:
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
package_directory = r'/content/drive/MyDrive/CS_546_Project_Dependencies/lib/python3.10/site-packages'

In [ ]:
import sys
#sys.path.append(package_directory)

In [ ]:
results_directory = r'/content/drive/MyDrive/CS_546_Project_Dependencies/RE_Finetuning'

In [ ]:
import pandas as pd
import numpy as np
import time

In [ ]:
from langchain.chains import LLMChain
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate,MessagesPlaceholder
from langchain_core.messages import SystemMessage
from langchain_groq import ChatGroq

In [ ]:
from groq import Groq

In [ ]:
import en_core_sci_md

## Dependency Issue resolution

- https://github.com/apple/turicreate/issues/3383

In [ ]:
!pip install langchain-groq --target=$package_directory

In [ ]:
%pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 7.6 MB/s eta 0:00:00


In [ ]:
import anyio

In [ ]:
!pip uninstall anyio -y

Found existing installation: anyio 3.7.1
Uninstalling anyio-3.7.1:
  Successfully uninstalled anyio-3.7.1


In [ ]:
!pip install anyio==3.5.0 --target=$package_directory

In [ ]:
!pip install h5py --target=$package_directory
!pip install typing-extensions --target=$package_directory
!pip install wheel --target=$package_directory

In [ ]:
!pip install openai==1.56.1 --target=$package_directory
#!pip install openai==1.55.3 --force-reinstall --quiet --use-deprecated=legacy-resolver --target=$package_directory
#!pip install httpx==0.27.2 -force-reinstall --quiet --use-deprecated=legacy-resolver --target=$package_directory

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.8/389.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.8/431.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.9/164.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
!pip install httpx==0.27.2 -force-reinstall --quiet --use-deprecated=legacy-resolver --target=$package_directory

In [ ]:
import os
os.kill(os.getpid(), 9)

## Implementation

In [ ]:
groq_api_key = "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" # Replace with environment variable

In [ ]:
current_model_index = 0

In [ ]:
models = ['llama3-8b-8192','llama-3.1-8b-instant','mixtral-8x7b-32768']

In [ ]:
groq_model = ChatGroq(
    api_key = groq_api_key,
    model = models[current_model_index],
    temperature = 0.15)

In [ ]:
conversational_memory_length = 3

In [ ]:
memory = ConversationBufferWindowMemory(k=conversational_memory_length, memory_key="chat_history", return_messages=True)

In [ ]:
memory

ConversationBufferWindowMemory(chat_memory=InMemoryChatMessageHistory(messages=[]), return_messages=True, memory_key='chat_history', k=3)

In [ ]:
base_message = '''
Extract the entity-relation-entity triplets from the input sentence. Here, the goal is to output the triplets in a template used by REBEL model and the format is given below.

Triplet Format:\n
<triplet> ENTITY 1 <subj> ENTITY 2 <obj> RELATION </triplet>\n
Head Entity: The entity between <triplet> and <subj>.\n
Tail Entity: The entity between <subj> and <obj>.\n
Example: <triplet> codon 1763 <subj> valine to methionine <obj> changed </triplet>\n

Here are some additional guidelines to keep in mind. All of them have to be followed at all times.\n

Entity Extraction:
1) Exclude articles and adjectives from entities to retain only the core information.
Example:
"A novel disease," "The novel disease," → disease.\n
2) If a head entity has multiple tail entities, include them in the same triplet, as shown below:\n
Examples:
General Template - <triplet> ENTITY 1 <subj> ENTITY 2 <obj> RELATION <subj> ENTITY 3 <obj> RELATION ... </triplet>\n
Real example - <triplet> boy <subj> paris <obj> born <subj> 22 years <obj> age <subj> singer <obj> profession </triplet>\n

NOTE - The above examples are just guidelines. In reality, there could be variable number of relations for a specific head entity. Approach it case by case.

Relation Extraction:
Extract relations directly from the text as spans.
Keep relations concise, including only the essential information. Avoid overly long or verbose spans.

Comprehensive Capture:
Ensure all relations associated with a head entity are included in a single <triplet> tag.

These rules ensure the extraction process captures key relationships from the text accurately and efficiently, adhering to the REBEL model's requirements.
'''

In [ ]:
base_message_1 = """
You are given a passage and a set of extracted triplets. Your task is to:

Modify existing relations in the triplets using spans extracted directly from the passage while keeping the same entities.
Identify and include additional triplets from the passage that are missing in the provided set. Use the same format for all triplets.

Instructions:
This follows the REBEL format for entity-relation extraction.
Use concise and essential relation spans from the passage for the relations.

Retain the structure of the triplets as:
<triplet> ENTITY 1 <subj> ENTITY 2 <obj> RELATION
If a head entity (ENTITY 1) has multiple tail entities (ENTITY 2), combine them in the same <triplet> tag.
Do not include a closing </triplet> tag. Simply begin a new triplet with <triplet> when necessary.
Relations should be extracted from the passage and replace existing relations located between <obj> and the succeeding tag.

Output Format:
Modify each triplet's relations based on spans from the passage.
Add any missing triplets based on the information in the passage.
Return the final list of triplets.
Example Output:

<triplet> ENTITY 1 <subj> ENTITY 2 <obj> RELATION
<triplet> ENTITY 1 <subj> ENTITY 3 <obj> RELATION 2
...
Now process the input and return the revised and expanded triplets in the specified REBEL format.
"""

In [ ]:
memory.chat_memory.add_user_message(base_message_1)

In [ ]:
chat_prompt_base = ChatPromptTemplate(
                messages=[
                    SystemMessage(content=base_message_1),
                    MessagesPlaceholder(variable_name="chat_history"),
                    HumanMessagePromptTemplate.from_template("Update the triplets according to the REBEL template. Passage and ground truth triplets are given as input. Use only the entities present in the triplets & refer to the passage to extract the relation between the two entities. The output should only update the relations enclosed between <obj> and the succeeding tag by keeping the entities intact. Do not return anything else.\n {query}")
                ]
            )

In [ ]:
chat_prompt = ChatPromptTemplate(
                messages=[
                    MessagesPlaceholder(variable_name="chat_history"),
                    HumanMessagePromptTemplate.from_template("Extract the triplets as mentioned according to the REBEL template. I only want the updated triplets. Do not return anything else.\n {query}")
                ]
            )

In [ ]:
def switch_model():
    global current_model_index, groq_model, classification_chain
    current_model_index += 1
    if current_model_index < len(models):
        groq_model = ChatGroq(
            api_key=groq_api_key,
            model = models[current_model_index],
            temperature=0.15
        )
        classification_chain = LLMChain(
                                        llm=groq_model,
                                        prompt=chat_prompt_base,
                                        memory=memory)
        print(f"Switched to model: {models[current_model_index]}")
    else:
        print("All models have reached their limits for today.")
        raise Exception("No more models available for processing.")

In [ ]:
def extract_triplets(query: str):
    global groq_model, classification_chain
    try:
        #response = classification_chain.run(input={"passage": passage, "triplets": triplets})
        response = classification_chain.run({"query": query})
        return response

    except Exception as e:
        if "rate limit" in str(e).lower():
            print("Rate limit reached for the current model. Switching models...")
            switch_model()
            response = extract_triplets(query)
            print("Updating classification chain...")
            classification_chain = LLMChain(
                                        llm=groq_model,
                                        prompt=chat_prompt,
                                        memory=memory)
            return response
        else:
            if '413' in str(e):
                print(f"Memory error received: {str(e)}. Retaining only top 3 messages.")
                memory.chat_memory.messages = memory.chat_memory.messages[:3]
                return extract_triplets(query)
            else:
                print(f"Received error: {str(e)}. Skipping to next query.")
            return 'Error'

In [ ]:
classification_chain = LLMChain(
    llm=groq_model,
    prompt=chat_prompt_base,
    memory=memory
)

In [ ]:
results_path = results_directory + '/Groq_BioRED_Triplets.csv'

In [ ]:
df = pd.read_csv(results_path)

In [ ]:
df = df[['Input','Triplets']]

In [ ]:
df['Extracted_Triplets'] = None

In [ ]:
df.head()

,Input,Triplets,Extracted_Triplets
0,a novel scn5a mutation manifests as a malignan...,<triplet> bradycardia <subj> Na(v)1.5 <obj> As...,None
1,allelic expression imbalance of human mu opioi...,<triplet> OPRM1 <subj> pain <obj> Association ...,None
2,genetic polymorphisms in the carbonyl reductas...,<triplet> cardiotoxic <subj> doxorubicinol <ob...,None
3,debrisoquine phenotype and the pharmacokinetic...,<triplet> beta-1 adrenoceptor <subj> metoprolo...,None
4,the first founder dguok mutation associated wi...,<triplet> dGK <subj> mitochondrial DNA depleti...,None


In [ ]:
df['Input'] = df['Input'].str.lower()

In [ ]:
first_query_index = 0

In [ ]:
passage, triplets = df['Input'][first_query_index], df['Triplets'][first_query_index]

In [ ]:
response = extract_triplets(passage + "\n"+triplets)

In [ ]:
response

'<triplet> a novel scn5a mutation <subj> manifests <obj> as a malignant form of long qt syndrome with perinatal onset of tachycardia/bradycardia. <triplet> congenital long qt syndrome (LQTS) <subj> with in utero onset of the rhythm disturbances <obj> is associated with a poor prognosis. <triplet> a newborn patient <subj> with fetal bradycardia, 2:1 atrioventricular block and ventricular tachycardia <obj> soon after birth. <triplet> 2:1 atrioventricular block <subj> improved to 1:1 conduction <obj> only after intravenous lidocaine infusion or a high dose of mexiletine, which also controlled ventricular tachycardia. <triplet> a novel scn5a mutation <subj> was identified <obj> in the transmembrane segment 6 of domain iv of the na(v)1.5 cardiac sodium channel, with a g-->a substitution at codon 1763, which changed a valine (gtg) to a methionine (atg). <triplet> the proband <subj> was heterozygous <obj> but a novel scn5a mutation was absent in the parents and the sister. <triplet> expressio

In [ ]:
df.loc[first_query_index,'Extracted_Triplets'] = response

In [ ]:
first_query_index += 1

In [ ]:
df['query'] = df['Input'] + "\n" + df['Triplets']

In [ ]:
query = df['query'][first_query_index]

In [ ]:
response = extract_triplets(query)

In [ ]:
response

'<triplet> OPRM1 <subj> pain <obj> plays a key role in\n<triplet> OPRM1 <subj> addiction <obj> predisposition\n<triplet> OPRM1 <subj> variant a118g <obj> caused\n<triplet> OPRM1 <subj> variant a118g <obj> leading to\n<triplet> OPRM1 <subj> mrna expression <obj> measured\n<triplet> OPRM1 <subj> mrna allele <obj> 1.5-2.5-fold more abundant\n<triplet> OPRM1 <subj> protein levels <obj> measured by western blotting and receptor binding assay\n<triplet> OPRM1 <subj> mrna stability <obj> differences\n<triplet> OPRM1 <subj> mrna maturation <obj> defect\n<triplet> OPRM1 <subj> protein yield <obj> deleterious effects'

In [ ]:
df.loc[first_query_index,'Triplets'] = response

In [ ]:
classification_chain = LLMChain(
    llm=groq_model,
    prompt=chat_prompt,
    memory=memory
)

In [ ]:
no_batches = 5

In [ ]:
batch_size = len(df) // no_batches
batch_results = []

In [ ]:
batch_size

20

In [ ]:
start = 0 # Initial value is 1
end = batch_size # Initial value is batch_size = 25

In [ ]:
for i in range(no_batches):
    print(f'Processing batch: {i+1}')

    df_batch = df.iloc[start:end].copy()
    df_batch['Extracted_Triplets'] = df_batch['query'].apply(lambda query: extract_triplets(query))

    df.iloc[start:end] = df_batch

    batch_results.append(df_batch)

    df.to_csv(results_path, index=False)
    start = end
    end = end + batch_size if i < no_batches-1 else len(df)

    time.sleep(60)

Processing batch: 1
Processing batch: 2
Processing batch: 3
Processing batch: 4
Processing batch: 5


In [ ]:
df['Extracted_Triplets'][0]

'<triplet> bradycardia <subj> Na(v)1.5 <obj> manifests\n<triplet> tachycardia <subj> Na(v)1.5 <obj> manifests\n<triplet> arrhythmias <subj> Na(v)1.5 <obj> dysfunction\n<triplet> arrhythmias <subj> mexiletine <obj> controlled\n<triplet> arrhythmias <subj> lidocaine <obj> controlled\n<triplet> LQTS <subj> Na(v)1.5 <obj> associated\n<triplet> V1763M <subj> arrhythmias <obj> associated\n<triplet> V1763M <subj> Na(v)1.5 <obj> associated\n<triplet> G-->A substitution at codon 1763 <subj> Na(v)1.5 <obj> changed\n<triplet> mexiletine <subj> ventricular tachycardia <obj> controlled\n<triplet> lidocaine <subj> ventricular tachycardia <obj> controlled\n<triplet> mexiletine <subj> atrioventricular block <obj> improved\n<triplet> lidocaine <subj> atrioventricular block <obj> improved'